# Banking FAQ RAG Pipeline

## Objective

Build a Retrieval-Augmented Generation (RAG) pipeline using:

- LangChain
- FAISS Vector Database
- HuggingFace Embeddings
- Groq Llama 3.3 LLM

This notebook will:
- load the FAISS vector database
- retrieve relevant banking FAQs
- generate AI-powered contextual responses
- build a complete RAG pipeline
- prepare for conversational banking chatbot deployment

---

## Technologies Used

- LangChain
- FAISS
- Groq LLM
- HuggingFace Embeddings
- Retrieval-Augmented Generation (RAG)

### Import Libraries

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# Environment Variables
import os

# LangChain
from langchain_community.vectorstores import FAISS

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Groq LLM
from langchain_groq import ChatGroq

# Prompt Templates
from langchain.prompts import PromptTemplate

# Retrieval Chains
from langchain.chains import RetrievalQA

# Load Environment Variables
from dotenv import load_dotenv

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

### Load Environment Variables

In [2]:
# Load .env File
load_dotenv()

# Get Groq API Key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Check API Key
print("API Key Loaded Successfully")

API Key Loaded Successfully


### What is RAG?

RAG stands for:

```text
Retrieval-Augmented Generation
```

Traditional LLM:
```text
Question → LLM → Answer
```

RAG Pipeline:
```text
Question
→ Retrieve Relevant Documents
→ Send Context to LLM
→ Generate Better Answer
```

---

# Why Use RAG?

LLMs alone:
- may hallucinate
- may give incorrect answers
- lack domain-specific knowledge

RAG:
- retrieves real banking data
- grounds responses in actual FAQs
- improves factual accuracy
- reduces hallucinations

---

# Banking RAG Benefits

For banking systems:
- higher accuracy
- contextual answers
- regulatory-safe responses
- enterprise AI architecture

### Load Embedding Model

In [3]:
def load_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

embeddings = load_embeddings()
print("Embedding model loaded successfully.")

Embedding model loaded successfully.


### Load FAISS Vector Database

In [4]:
vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index", embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS vector database loaded successfully.")

FAISS vector database loaded successfully.


### Create Retriever

In [5]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created successfully.")

Retriever created successfully.


### Test Retriever

In [6]:
query = "How can I transfer money internationally?"

retrieved_docs = retriever.invoke(query)

# Total Documents
print("Retrieved Documents:", len(retrieved_docs))

Retrieved Documents: 5


#### View Retrieved Documents

In [7]:
for i, doc in enumerate(retrieved_docs):
    print("="*80)
    print(f"Document {i+1}")
    print("="*80)

    print(doc.page_content)
    print("\n")

Document 1
Question: Give me details about how do i transfer money to another bank.
Answer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Document 2
Question: Tell me about how do i transfer money to another bank.
Answer: Usually, Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Document 3
Question: Could you describe How do I transfer money to another bank? #47062
Answer: Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Document 4
Question: What do you mean by how do i transfer money to another bank?
Answer: In simple words, Use NEFT, RTGS, or IMPS through your mobile banking app or net banking. Just enter the recipient's account number, IFSC code, and the amount.


Document 5
Question: Could you descr

### Why Groq Llama 3.3?

We are using:

```text
llama-3.3-70b-versatile
```

through Groq API.

Advantages:
- extremely fast inference
- powerful reasoning
- large context understanding
- excellent conversational quality
- ideal for RAG pipelines

---

# Why Groq for This Project?

Compared to local LLMs:
- lower laptop memory usage
- no GPU required
- faster responses
- enterprise-quality performance

#### Load Groq LLM

In [8]:
def load_llm():
    llm = ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name="llama-3.3-70b-versatile",
        temperature=0
    )

    return llm

llm = load_llm()

print("Groq LLM loaded successfully.")

Groq LLM loaded successfully.


#### Test LLM

In [9]:
response = llm.invoke("What is a savings account?")

print(response.content)

A savings account is a type of deposit account offered by banks and credit unions that allows individuals to store their money securely while earning interest on their deposits. The primary purpose of a savings account is to provide a safe and liquid place to save money for short-term or long-term goals, such as emergencies, large purchases, or retirement.

Here are some key features of a savings account:

1. **Interest earnings**: Savings accounts typically earn interest on the deposited amount, which can help grow your savings over time.
2. **Liquidity**: Savings accounts are liquid, meaning you can access your money when needed, although some accounts may have restrictions or penalties for early withdrawals.
3. **Low risk**: Savings accounts are generally considered low-risk investments, as they are insured by government agencies such as the Federal Deposit Insurance Corporation (FDIC) or the National Credit Union Administration (NCUA).
4. **Limited transactions**: Savings accounts 

#### Prompt Engineering

##### Why Prompt Engineering?

Prompt templates help:
- control LLM behavior
- reduce hallucinations
- improve response quality
- enforce banking-safe responses

The LLM will:
- answer only from retrieved context
- avoid making up information
- behave like a banking assistant

In [10]:
prompt_template = """

You are an AI Banking Assistant.

Use ONLY the provided banking context to answer the user's question.

If the answer is not available in the context,
say:
"I could not find relevant banking information."

Answer clearly and professionally.

Context:
{context}

Question:
{question}

Answer:

"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=[
        "context",
        "question"
    ]
)

print("Prompt Template Created Successfully")

Prompt Template Created Successfully


#### Build RAG Chain

In [11]:
# Create RetrievalQA Chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type= "stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs= {"prompt": PROMPT}
)

print("RAG Chain Created Successfully")

RAG Chain Created Successfully


##### Test RAG Pipeline

In [12]:
# Example Query 1
query = "How can I block my debit card?"

response = rag_chain.invoke(
    {"query": query}
)

print(response["result"])

Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.


In [13]:
# Example Query 2
query = "Explain EMI payments."

response = rag_chain.invoke(
    {"query": query}
)

print(response["result"])

EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every month to repay your loan, combining principal and interest.


#### Understanding RAG Pipeline Flow

The system now follows:

```text
User Question
↓
Embedding Generation
↓
FAISS Similarity Search
↓
Retrieve Relevant Banking FAQs
↓
Send Context to LLM
↓
Generate AI Response
```

This architecture is used in:
- ChatGPT retrieval systems
- enterprise AI assistants
- banking copilots
- customer support AI

#### Display Retrieved Sources

In [14]:
source_docs = response["source_documents"]

for i, doc in enumerate(source_docs):
    print("="*80)
    print(f"Source Document {i+1}")
    print("="*80)

    print(doc.page_content)
    print("\n")

Source Document 1
Question: Could you explain an emi?
Answer: EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every month to repay your loan, combining principal and interest.


Source Document 2
Question: Please explain an emi.
Answer: Generally, EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every month to repay your loan, combining principal and interest.


Source Document 3
Question: I want to understand an emi.
Answer: EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every month to repay your loan, combining principal and interest.


Source Document 4
Question: Help me understand an emi.
Answer: In simple words, EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every month to repay your loan, combining principal and interest.


Source Document 5
Question: Tell me about what is an emi
Answer: Basically, EMI stands for Equated Monthly Instalment. It's the fixed amount you pay every mont

#### Test Another Banking Query

In [15]:
query = "What happens if I miss my EMI payment?"

response = rag_chain.invoke(
    {"query": query}
)

print(response["result"])

I could not find relevant banking information.


#### Test Financial Query

In [16]:
query = "Explain mutual funds"

response = rag_chain.invoke(
    {"query": query}
)

print(response["result"])

A mutual fund pools money from many investors and invests it in stocks, bonds, or other securities managed by a professional fund manager.


### Compare Semantic Search vs RAG

#### Semantic Search vs RAG

| Semantic Search | RAG |
|---|---|
| retrieves documents | retrieves + generates |
| no AI generation | contextual AI response |
| search engine | conversational assistant |
| returns FAQs | generates natural answers |
| retrieval only | retrieval + generation |

---

#### Why RAG is Powerful

RAG combines:
- semantic retrieval
- contextual understanding
- generative AI

This creates:
- intelligent AI assistants
- domain-aware chatbots
- enterprise conversational systems

# Advantages of RAG

RAG provides:

- grounded AI responses
- reduced hallucinations
- domain-specific intelligence
- scalable knowledge retrieval
- explainable AI answers

This architecture is widely used in:
- enterprise chatbots
- AI assistants
- banking AI systems
- legal AI systems
- healthcare AI systems

# Current Limitations

Current RAG system still has some limitations:

- no conversational memory
- no chat history
- no streaming responses
- no guardrails
- no reranking
- no hybrid retrieval

Future improvements:
- conversational memory
- hybrid search
- reranking models
- chatbot UI
- streaming generation

#### Create Production RAG Function

In [17]:
def ask_banking_assistant(query):
    response = rag_chain.invoke(
        {"query": query}
    )

    return {
        "question": query,
        "answer": response["result"],
        "source_documents": response["source_documents"]
    }

#### Final RAG Testing

In [18]:
query = "How to activate mobile banking?"

result = ask_banking_assistant(query)

print("Question:")
print(result["question"])

print("\n")

print("Answer:")
print(result["answer"])

Question:
How to activate mobile banking?


Answer:
Download your bank's official app, enter your account details, verify with an OTP on your registered mobile, and set your MPIN.


### Save RAG Pipeline Configuration

In [19]:
# Save Configuration

rag_config = {

    "embedding_model":
    "sentence-transformers/all-MiniLM-L6-v2",

    "llm_model":
    "llama-3.3-70b-versatile",

    "vector_database":
    "FAISS",

    "retrieval_k":
    5
}

print(rag_config)

{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2', 'llm_model': 'llama-3.3-70b-versatile', 'vector_database': 'FAISS', 'retrieval_k': 5}


In [20]:
import joblib

joblib.dump(
    retriever,
    "../models/rag/retriever.pkl"
)

print("Retriever saved successfully.")

Retriever saved successfully.


### Banking AI Architecture

The project now includes:

---

# Phase 1 — EDA
✅ completed

# Phase 2 — NLP + Intent Classification
✅ completed

# Phase 3 — Semantic Search
✅ completed

# Phase 4 — FAISS Vector Database
✅ completed

# Phase 5 — RAG Pipeline
✅ completed

---

# Current AI Pipeline

```text
User Query
↓
Sentence Embedding
↓
FAISS Vector Retrieval
↓
Relevant Banking FAQs
↓
Groq Llama 3.3
↓
AI Generated Banking Response
```

This is now a:
- modern GenAI system
- enterprise RAG architecture
- conversational banking AI assistant

# Key Insights & Observations

## 1. RAG Pipeline Successfully Implemented

The project now supports:
- Retrieval-Augmented Generation
- semantic retrieval
- contextual AI response generation
- banking-aware conversational AI

---

## 2. FAISS + LangChain Integration

The system uses:
- LangChain retrievers
- FAISS vector database
- HuggingFace embeddings
- Groq LLM integration

This creates a scalable enterprise AI architecture.

---

## 3. Hallucination Reduction

Instead of generating answers blindly,
the LLM now:
- retrieves banking FAQs
- uses real banking context
- generates grounded responses

This significantly reduces hallucinations.

---

## 4. Lightweight Yet Powerful Architecture

The project remains lightweight because:
- embeddings are compact
- FAISS is efficient
- Groq handles LLM inference remotely

This makes the system:
- laptop-friendly
- scalable
- deployment-ready

---

## 5. Enterprise-Level AI Pipeline

The architecture now resembles:
- enterprise RAG systems
- fintech AI assistants
- banking copilots
- customer support AI systems

---

## 6. Next Phase

Next notebook:
```text
08_generative_chatbot.ipynb
```

Will implement:
- conversational chatbot memory
- chat history
- streaming responses
- conversational banking assistant
- Streamlit GenAI interface